<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/baristabot_langgraph_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Défi quotidien : Créer un agent avec LangGraph et Gemini

Ce notebook implémente **BaristaBot**, un agent conversationnel de commande de café avec :

- un état de conversation `OrderState`
- un chatbot basé sur Gemini via LangChain
- un graphe LangGraph avec boucles conversationnelles
- un outil de menu dynamique `get_menu`
- des outils de commande : ajout, confirmation, lecture, suppression et validation de commande
- un routage conditionnel entre `chatbot`, `human`, `tools`, `ordering` et `END`

> Avant d’exécuter le notebook, ajoute ta clé API Gemini dans les secrets Colab sous le nom `GOOGLE_API_KEY`.

## 1. Installation des dépendances

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

## 2. Configuration de la clé API Gemini

In [ ]:
import os
import getpass

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# Option recommandée dans Google Colab : secret nommé GOOGLE_API_KEY
if not GOOGLE_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        GOOGLE_API_KEY = None

# Option Kaggle si le notebook est exécuté dans Kaggle
if not GOOGLE_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        GOOGLE_API_KEY = None

# Dernier recours : saisie manuelle
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass.getpass("Entre ta clé GOOGLE_API_KEY : ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Clé API configurée.")

## 3. Définition de l’état, des instructions système et du modèle Gemini

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages.ai import AIMessage
from IPython.display import Image, display


class OrderState(TypedDict):
    """State representing the customer's order conversation."""

    # Historique des messages.
    # add_messages signifie que les nouveaux messages sont ajoutés à l'historique.
    messages: Annotated[list, add_messages]

    # Commande en cours.
    order: list[str]

    # Indique si la commande est terminée.
    finished: bool


BARISTABOT_SYSINT = (
    "system",
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user). "
    "Always confirm_order with the user before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, call confirm_order to ensure it is correct, then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!"
)

WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

# Modèle Gemini utilisé par LangChain.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")

## 4. Définir un chatbot à un seul tour

In [ ]:
def chatbot(state: OrderState) -> OrderState:
    """The chatbot itself. A simple wrapper around the model's own chat interface."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition.
graph_builder = StateGraph(OrderState)

# Add the chatbot function to the app graph as a node called "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

chat_graph = graph_builder.compile()

## 5. Visualiser le premier graphe

In [ ]:
Image(chat_graph.get_graph().draw_mermaid_png())

## 6. Exécuter le graphe sur un premier message utilisateur

In [ ]:
from pprint import pprint

user_msg = "Hello, what can I order here?"

state = chat_graph.invoke({
    "messages": [("user", user_msg)],
    "order": [],
    "finished": False
})

for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

## 7. Ajouter un deuxième tour de conversation

In [ ]:
user_msg = "Do you have tea or only coffee?"

state["messages"].append(("user", user_msg))
state = chat_graph.invoke(state)

for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

## 8. Ajouter un nœud humain et une boucle conversationnelle

In [ ]:
def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    finished = state.get("finished", False)
    if user_input.lower().strip() in {"q", "quit", "exit", "goodbye"}:
        finished = True

    return state | {"messages": [("user", user_input)], "finished": finished}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot itself. A wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}


# Start building a new graph.
graph_builder = StateGraph(OrderState)

# Add the chatbot and human nodes to the app graph.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# Start with the chatbot again.
graph_builder.add_edge(START, "chatbot")

# The chatbot will always go to the human next.
graph_builder.add_edge("chatbot", "human")

## 9. Ajouter une condition de sortie

In [ ]:
def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Image(chat_with_human_graph.get_graph().draw_mermaid_png())

### Test optionnel du chatbot avec nœud humain

In [ ]:
# Décommente cette cellule si tu veux tester la boucle conversationnelle simple.
# Tape q, quit, exit ou goodbye pour arrêter.

# state = chat_with_human_graph.invoke(
#     {"messages": [], "order": [], "finished": False},
#     {"recursion_limit": 20}
# )
# pprint(state)

## 10. Ajouter un menu dynamique avec un outil

In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:

    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: Whole
    Espresso shots: Single, Double, Triple, Quadruple; Default: Double
    Caffeine: Decaf, Regular; Default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners: vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: extra hot, one pump, half caff, extra foam, etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like Dirty Chai Latte.
    "Regular milk" is the same as Whole milk.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
    """

In [ ]:
from langgraph.prebuilt import ToolNode

# Define the tools and create a "tools" node.
tools = [get_menu]
tool_node = ToolNode(tools)

# Attach the tools to the model so that it knows what it can call.
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Route between human or tool nodes, depending if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Add the nodes, including the new tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Chatbot may go to tools, or human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)

# Human may go back to chatbot, or exit.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

Image(graph_with_menu.get_graph().draw_mermaid_png())

### Test optionnel du chatbot avec menu

In [ ]:
# Décommente pour tester le bot avec l'outil get_menu.
# Exemple à demander : "What is on the menu?"

# state = graph_with_menu.invoke(
#     {"messages": [], "order": [], "finished": False},
#     {"recursion_limit": 30}
# )
# pprint(state)

## 11. Gérer les commandes avec des outils de commande

In [ ]:
from collections.abc import Iterable
from random import randint
from langchain_core.messages.tool import ToolMessage


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """
    pass


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """
    pass


@tool
def get_order() -> str:
    """Returns the user's order so far. One item per line."""
    pass


@tool
def clear_order() -> str:
    """Removes all items from the user's order."""
    pass


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """
    pass


def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    tool_msg = state["messages"][-1]
    order = list(state.get("order", []))
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call.get("args", {})

        if tool_name == "add_to_order":
            drink = tool_args["drink"]
            modifiers = tool_args.get("modifiers", [])

            if isinstance(modifiers, str):
                modifiers = [modifiers]

            modifiers = list(modifiers)
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f"{drink} ({modifier_str})")
            response = "Updated order:\\n" + "\\n".join(order)

        elif tool_name == "confirm_order":
            print("Your order:")
            if not order:
                print("  (no items)")

            for drink in order:
                print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_name == "get_order":
            response = "\\n".join(order) if order else "(no order)"

        elif tool_name == "clear_order":
            order.clear()
            response = "Order cleared."

        elif tool_name == "place_order":
            if not order:
                response = "No order to place. Please add at least one item first."
                order_placed = False
            else:
                order_text = "\\n".join(order)
                print("Sending order to kitchen!")
                print(order_text)

                order_placed = True
                response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f"Unknown tool call: {tool_name}")

        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_name,
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}


def maybe_route_to_tools(state: OrderState) -> str:
    """Route between chat and tool nodes if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Route to `tools` node for automated tool calls first.
        if any(tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

## 12. Définir le graphe complet avec outils automatiques et outils de commande

In [ ]:
# Auto-tools will be invoked automatically by the ToolNode.
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


def chatbot_with_all_tools(state: OrderState) -> OrderState:
    """Chatbot connected to both menu tools and order tools."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_all_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)

# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")

graph_with_order_tools = graph_builder.compile()

Image(graph_with_order_tools.get_graph().draw_mermaid_png())

## 13. Exécuter le système complet de commande

In [ ]:
# The default recursion limit for traversing nodes is 25.
# Setting it higher lets you try a more complex order with multiple steps.
config = {"recursion_limit": 100}

state = graph_with_order_tools.invoke(
    {"messages": [], "order": [], "finished": False},
    config
)

# Things to try:
# - Order a drink.
# - Make a change to your order.
# - Ask: "Which teas are from England?"
# - Try to order soy milk and observe how the bot handles unavailable items.

pprint(state)

## Conclusion

Dans ce notebook, nous avons construit un agent LangGraph complet avec :

- un état partagé entre les nœuds
- une boucle conversationnelle humain ↔ chatbot
- un outil automatique pour consulter le menu
- un nœud séparé pour manipuler l’état de la commande
- un routage conditionnel vers les bons nœuds
- une sortie propre après validation de la commande

Ce projet montre comment LangGraph permet de modéliser un agent IA comme un vrai système applicatif avec mémoire, outils, logique métier et transitions contrôlées.